# Tutorial 13 · Markov Chains & the n-gram LM — Part B

**~40 minutes · after the worksheet**

Compute a stationary distribution, fit smoothed character transitions, score held-out text, and generate a sample.

Work in pairs. Before each code cell, write down the qualitative result you expect. The notebook is designed to run top-to-bottom in a fresh kernel.


## 1 · Power iteration finds the stationary distribution


In [1]:
import collections, math, string
import numpy as np

P=np.array([[.8,.2],[.4,.6]]); mu=np.array([1.,0.])
for _ in range(30): mu=mu@P
print(mu, "residual",np.linalg.norm(mu@P-mu))


[0.66666667 0.33333333] residual 3.2610820366862006e-13


## 2 · Fit a smoothed character bigram


In [2]:
from pathlib import Path

def clean(text):
    text=text.lower(); return " ".join("".join(ch if ch in string.ascii_lowercase else " " for ch in text).split())
source = Path("lecture-plan-detailed.md")
if not source.exists(): source = Path("../lecture-plan-detailed.md")
text=clean(source.read_text())
split=int(.8*len(text)); train,test=text[:split],text[split:]
alphabet=" "+string.ascii_lowercase; K=len(alphabet); alpha=.1
counts=collections.defaultdict(collections.Counter)
for a,b in zip(train[:-1],train[1:]): counts[a][b]+=1
def row(context):
    total=sum(counts[context].values())+alpha*K
    return np.array([(counts[context][ch]+alpha)/total for ch in alphabet])
loss=[]
for a,b in zip(test[:-1],test[1:]): loss.append(-math.log2(row(a)[alphabet.index(b)]))
print("held-out BPC",np.mean(loss),"perplexity",2**np.mean(loss))


held-out BPC 3.580975897764388 perplexity 11.966886141682448


## 3 · Generate from the same rows


In [3]:
rng=np.random.default_rng(13); current="t"; output=[current]
for _ in range(400):
    current=rng.choice(list(alphabet),p=row(current)); output.append(current)
print("".join(output))


trss aur ty s ithe sd ry qw mubaprin quawlere morivsthanided invautieralinea gps mbexplgrernonkncscondin s murillyerealeth l omumlinvate crtpeick aktil t acruarin pivy beothentarindus ain fororemoonsts bes riz s onaine ibineinf d zerane d qus llit panedib une liumaterowains it b l geme stily tiken kke diaseckacthictoksthat vs ulos gtrmat e an whon i epati ivacheceefoy acht onarorad d laryn umaiain 


## 4 · Closing experiment

Change `alpha`, the training fraction, or the context order. Report held-out BPC before judging samples by eye.


## Closing check

Write three sentences: one numerical result you verified, one geometric/probabilistic interpretation, and one failure mode you would now test in a larger implementation.
